Aquí te dejo el código para trabajar con imágenes satelitales en formato GeoTIFF usando rasterio para leer los datos y luego procesarlos con PyTorch. Este código es útil si tienes imágenes geoespaciales y deseas clasificarlas con una red neuronal convolucional.

In [ ]:
# Importación de librerías
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms

In [ ]:
# Función para leer imágenes GeoTIFF
def load_geotiff(image_path):
    with rasterio.open(image_path) as src:
        img_array = src.read([1, 2, 3])  # Leer las bandas RGB
        img_array = np.moveaxis(img_array, 0, -1)  # Cambiar el orden para que sea (height, width, channels)
    return img_array

In [ ]:
# Crear una clase Dataset personalizada para GeoTIFF
class GeoTIFFDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = load_geotiff(self.image_paths[idx])
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
# Definir transformaciones
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Suponiendo que tienes listas de rutas de imágenes y etiquetas
image_paths = ['path/to/image1.tif', 'path/to/image2.tif']  # Rutas a las imágenes GeoTIFF
labels = [0, 1]  # Etiquetas correspondientes

In [ ]:
# Crear el DataLoader
dataset = GeoTIFFDataset(image_paths, labels, transform)
data_loader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
# Definir el modelo CNN
model = models.resnet18(pretrained=True)  # Usamos ResNet preentrenada
model.fc = nn.Linear(model.fc.in_features, 2)  # Cambiar la capa final para clasificación binaria

In [ ]:
# Definir el optimizador y la función de pérdida
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Entrenar el modelo
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in data_loader:
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(data_loader)}")

In [ ]:
# Evaluar el modelo (en este caso no tenemos un conjunto de validación)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in data_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

In [ ]:
print(f'Accuracy on dataset: {100 * correct / total}%')